<a href="https://colab.research.google.com/github/ribkhinaura/KKA2-Proyek-Modul-EDA/blob/main/Proyek_Modul_KKA_2_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📋 D. Lembar Perencanaan Proyek Kelompok (Project Charter)

* **Mata Pelajaran:** Koding dan Kecerdasan Artificial (KKA 2 -EDA) **Kelas: XI RPL 2**

* **Nama*: Ribkhi Fadhilatun Naura* *No. Absen: 32*

* **Nama*: Amira Fawwaza Farihan Asadela *No. Absen: 3*

* **Dataset yang Dipilih:** Data Penjualan Kantin Sekolah
* **Pertanyaan Analisis Awal:**
  1. Menu apa yang memberikan kontribusi pendapatan terbesar bagi kantin?
  2. Bagaimana sebaran tingkat penjualan item (apakah ada menu yang tergolong sangat laris)?
  3. Berapa total pendapatan bersih kantin setelah data dibersihkan?
* **Dugaan Masalah Kualitas Data:**
  - Missing value pada kolom `terjual` dan `menu`
  - Duplikat data pada beberapa baris transaksi
  - Tipe data `harga` atau `terjual` yang tidak sesuai (misal float/string padahal integer)
* **Rencana Teknik Pembersihan:**
  - `fillna(0)` untuk data `terjual` yang kosong (mengasumsikan belum ada penjualan)
  - `dropna(subset=['menu'])` untuk menghapus data tanpa nama menu
  - `drop_duplicates()` untuk menghapus baris transaksi ganda
  - `.astype(int)` untuk merapikan tipe data numerik
* **Rencana Manipulasi Data:**
  - **Kolom Turunan:** Membuat kolom `total_pendapatan` (`harga` * `terjual`)
  - **Filter:** Menyaring menu laris (`terjual` > 20)
  - **Sort:** Mengurutkan menu dari pendapatan tertinggi
  - **Groupby/Agregasi:** Menghitung total pendapatan per kategori `menu`
* **Pembagian Peran:**
  - **Anggota 1:** Data Loading, Inspection, & Data Cleaning
  - **Anggota 2:** Data Manipulation, Profiling Summary, & Exporting Dataset

### Data Loading & Inspection

Memuat dataset ke dalam Pandas DataFrame dan melakukan inspeksi awal untuk memahami struktur data, jumlah baris/kolom, tipe data.

In [ ]:
import numpy as np
import pandas as pd

# 1. Membuat/Memuat Dataset (Simulasi data mentah kantin sesuai latihan modul)
# Catatan: Jika memiliki file CSV eksternal, gunakan pd.read_csv('nama_file.csv')
raw_data = {
    'menu': ['Nasi Goreng', 'Es Teh', 'Mie Ayam', 'Es Teh', None, 'Nasi Goreng', 'Ayam Geprek', 'Es Jeruk'],
    'kategori': ['Makanan', 'Minuman', 'Makanan', 'Minuman', 'Makanan', 'Makanan', 'Makanan', 'Minuman'],
    'harga': [12000, 4000, 10000, 4000, 8000, 12000, 15000, 5000],
    'terjual': [23.0, 40.0, None, 35.0, 18.0, 23.0, 12.0, None]
}

df = pd.DataFrame(raw_data)

# Menyimpan ke CSV mentah untuk simulasi loading
df.to_csv('data_kantin_raw.csv', index=False)

# --- Awal Data Loading & Inspection ---
print("=== 1. HEAD (5 Baris Pertama) ===")
display(df.head())

print("\n=== 2. INFO DATASET (Tipe Data & Non-Null) ===")
df.info()

print("\n=== 3. STATISTIK DESKRIPTIF (DESCRIBE) ===")
display(df.describe())

print("\n=== 4. DIMENSI DATASET (SHAPE) ===")
print(f"Jumlah Baris: {df.shape[0]} | Jumlah Kolom: {df.shape[1]}")

### Data Cleaning

#### Catatan Alasan Teknik Pembersihan:
1. **Handling Missing Value**:
   - Kolom `terjual`: Diisi dengan angka `0` menggunakan `.fillna(0)` karena data kosong pada jumlah penjualan secara logis diasumsikan belum ada transaksi/terjual 0 item.
   - Kolom `menu`: Dihapus menggunakan `.dropna(subset=['menu'])` karena nama menu adalah entitas utama. Data tanpa nama menu tidak dapat dianalisis secara valid.
2. **Handling Duplikat**:
   - Baris duplikat dihapus menggunakan `.drop_duplicates()` agar tidak terjadi penganalisisan ganda pada transaksi yang sama.
3. **Penyesuaian Tipe Data**:
   - Kolom `terjual` diubah dari `float64` menjadi `int64` karena item barang yang dijual bernilai bulat (diskrit).

In [ ]:
print("=== IDENTIFIKASI MASALAH ===")
print("Jumlah Missing Value per Kolom:")
print(df.isnull().sum())
print(f"\nJumlah Baris Duplikat: {df.duplicated().sum()}")

# 1. Menangani Missing Value
df['terjual'] = df['terjual'].fillna(0)
df = df.dropna(subset=['menu'])

# 2. Menangani Duplikat
df = df.drop_duplicates().reset_index(drop=True)

# 3. Penyesuaian Tipe Data
df['terjual'] = df['terjual'].astype(int)
df['harga'] = df['harga'].astype(int)

print("\n=== DATASET SETELAH DIBERSIHKAN ===")
display(df)
print("\nVerifikasi Tipe Data & Missing Value Baru:")
df.info()

## Data Manipulation

### Operasi yang Dilakukan:
1. **Kolom Turunan:** Membuat kolom `total_pendapatan` = `harga` * `terjual`.
2. **Filtering:** Menyaring menu yang termasuk kategori laris (`terjual` > 20).
3. **Sorting:** Mengurutkan data berdasarkan `total_pendapatan` secara *descending* (tertinggi ke terendah).
4. **GroupBy / Agregasi:** Mengelompokkan data berdasarkan `kategori` dan `menu` untuk menghitung total item terjual dan total pendapatan.

In [ ]:
# 1. Kolom Turunan
df['total_pendapatan'] = df['harga'] * df['terjual']

# 2. Filtering (Menu Laris > 20)
menu_laris = df[df['terjual'] > 20]

# 3. Sorting (Berdasarkan Total Pendapatan Tertinggi)
df_sorted = df.sort_values(by='total_pendapatan', ascending=False).reset_index(drop=True)

# 4. GroupBy / Agregasi
ringkasan_menu = df.groupby('menu')[['terjual', 'total_pendapatan']].sum().sort_values(by='total_pendapatan', ascending=False).reset_index()
ringkasan_kategori = df.groupby('kategori')[['terjual', 'total_pendapatan']].sum().reset_index()

print("=== 1. HASIL DATAFRAME DENGAN KOLOM TURUNAN & SORTING ===")
display(df_sorted)

print("\n=== 2. FILTERING: MENU LARIS (TERJUAL > 20) ===")
display(menu_laris)

print("\n=== 3. GROUPBY: RINGKASAN PENDAPATAN PER MENU ===")
display(ringkasan_menu)

print("\n=== 4. GROUPBY: RINGKASAN PENDAPATAN PER KATEGORI ===")
display(ringkasan_kategori)

### Simpan Dataset Bersih

Dataset yang telah dibersihkan dan diolah disimpan ke format CSV sebagai bahan siap pakai untuk tahapan visualisasi data selanjutnya.

In [ ]:
# Menyimpan dataset bersih
nama_file_bersih = 'dataset_bersih.csv'
df.to_csv(nama_file_bersih, index=False)

print(f"✅ Dataset bersih berhasil disimpan dengan nama: '{nama_file_bersih}'")

## Lembar Refleksi Proyek

1. **Tahap yang Paling Menantang & Cara Mengatasinya:**
   - Tahap **Data Cleaning** adalah yang paling menantang karena kami harus menentukan strategi penanganan data hilang (`fillna` vs `dropna`) tanpa merusak integritas data asli. Kami mengatasinya dengan mendiskusikan logika bisnis (misal: jika nama menu hilang, data tidak berguna; namun jika angka penjualan hilang, dapat diasumsikan 0).
2. **Pentingnya Alasan Logis dalam Pembersihan Data:**
   - Keputusan pembersihan data tidak boleh dilakukan asal-asalan karena akan berdampak langsung pada keakuratan hasil analisis numerik. Menghapus data secara sembarangan bisa mereduksi informasi penting, sedangkan mengisi angka salah dapat bias pada nilai rata-rata/total.
3. **Hubungan Dataset Bersih dengan Pekerjaan Data Analyst:**
   - Di dunia nyata, mayoritas waktu seorang *Data Analyst* dihabiskan untuk *Data Cleaning* dan *Preprocessing*. Dataset bersih adalah fondasi utama sebelum melangkah ke pemodelan Machine Learning, Dashboarding, atau pengambilan keputusan bisnis yang krusial.